<a href="https://colab.research.google.com/github/imad2014/Analyse-du-Risque-de-Sinistre-Auto-Portefeuille-Marocain/blob/main/car_inssurence_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

0. Imports et configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

##  1. Chargement et Nettoyage des Données
### 1.1 Chargement

In [ ]:
data = pd.read_csv('/content/assurance_auto_raw.csv')

### 1.2 Rapport de qualité des données

In [ ]:
def rapport_data(df):
  info = pd.DataFrame({
      'type'   : df.dtypes,
      'Null'   : df.isna().sum(),
      '% Null' : (df.isnull().sum()/len(data))*100,
      'Uniques': df.nunique(),
  })
  return info
print ('              === RAPPORT QUALITE DONNES BRUT ===')
rapport = rapport_data(data)
print(rapport.to_string())
print('\n--- nnombre de colonnes Doublontes:{}'.format(data.duplicated().sum()))
print('--- IDs dupliqués : {}'.format(data['id_police'].duplicated().sum()))


              === RAPPORT QUALITE DONNES BRUT ===
                                   type  Null    % Null  Uniques
id_police                        object     0  0.000000     1000
age_conducteur                    int64     0  0.000000       62
genre                            object     0  0.000000        9
ville                            object     0  0.000000       13
categorie_socioprofessionnelle   object    47  4.630542        6
anciennete_permis_ans           float64    55  5.418719       52
marque_vehicule                  object     0  0.000000        8
age_vehicule_ans                float64    80  7.881773       23
puissance_fiscale_cv            float64    30  2.955665        9
usage_vehicule                   object     0  0.000000        3
km_annuels_estimes              float64    95  9.359606        9
formule_assurance                object     0  0.000000        6
coefficient_bonus_malus         float64    59  5.812808       91
nb_sinistres_anterieurs           int64 

### 1.3 Suppression des doublons

In [ ]:
# Supprimer les lignes entièrement dupliquées
data_F =  data.copy()
nbr_dup = data_F.duplicated().sum()
data_F.drop_duplicates(inplace=True)
print('doublons supprimés :{} \nlignes restantes : {}'.format(nbr_dup,len(data_F)))

doublons supprimés :10 
lignes restantes : 1005


In [ ]:
# Garder la première occurrence pour les IDs dupliqués
nbr_dup_id = data_F['id_police'].duplicated().sum()
data_F.drop_duplicates(subset=['id_police'],keep='first',inplace=True)
print ('IDs dupliqués supprimés:{}\nlignes restantes :{} '.format(nbr_dup_id,len(data_F)))


IDs dupliqués supprimés:5
lignes restantes :1000 


### 1.4 Correction des erreurs de saisie (standardisation)

In [ ]:
def unique_values_incategies_data(df):
  data_F_object_list = []
  for col in df.columns:
    if df[col].dtype == 'object':
      data_F_object_list.append(col)
  data_F_object_list.remove(data_F_object_list[0])
  data_F_object = data_F[data_F_object_list]
  for col in data_F_object.columns:
    print('{} : {}'.format(col,data_F_object[col].unique()))
unique_values_incategies_data(data_F)

genre : ['Homme' 'Femme' 'homme' 'Homme ' ' Homme' 'H' 'femme' 'FEMME' 'F']
ville : ['Marrakech' 'Oujda' 'Casablanca' 'Fes' 'Rabat' 'Tanger' ' Tanger'
 'Meknes' 'Agadir' 'marrakech' 'fes ' 'RABAT' 'Fès']
categorie_socioprofessionnelle : ['Commercant' 'Employe' 'Cadre' 'Independant' 'Etudiant' 'Retraite' nan]
marque_vehicule : ['Renault' 'Peugeot' 'Dacia' 'Volkswagen' 'Toyota' 'Hyundai' 'Fiat' 'Ford']
usage_vehicule : ['Personnel' 'Domicile_Travail' 'Professionnel']
formule_assurance : ['Tiers_Etendu' 'Tous_Risques' 'RC' 'rc' 'tous risques' 'tiers etendu']


In [ ]:
# --- Genre ---

data_F['genre'] = data_F['genre'].str.strip().str.capitalize()
data_F['genre'] = data_F['genre'].replace({'H':'Homme','F':'Femme'})

data_F.loc[data_F['genre'].str.startswith('H',na = False),'genre'] = 'Homme'
data_F.loc[data_F['genre'].str.startswith('F',na = False),'genre'] = 'Femme'
print('Valeurs genre après :', data_F['genre'].unique())

# --- Ville ---

data_F['ville'] = data_F['ville'].str.strip().str.capitalize()
data_F['ville'] = data_F['ville'].replace({'Fès':'Fes'})
print('Valeurs Ville après :',sorted(data_F['ville'].unique()))

# --- Formule ---
data_F['formule_assurance'] = data_F['formule_assurance'].str.strip().str.upper()
data_F['formule_assurance'] = data_F['formule_assurance'].replace({
      'TIERS ETENDU':'TIERS_ETENDU',
      'TOUS RISQUES':'TOUS_RISQUES',
      'RC' : 'RC'
})
print('Valeurs Formule après :',sorted(data_F['formule_assurance'].unique()))

Valeurs genre après : ['Homme' 'Femme']
Valeurs Ville après : ['Agadir', 'Casablanca', 'Fes', 'Marrakech', 'Meknes', 'Oujda', 'Rabat', 'Tanger']
Valeurs Formule après : ['RC', 'TIERS_ETENDU', 'TOUS_RISQUES']


### 1.5 Traitement des valeurs aberrantes

In [ ]:
def numerical_data(df):
  data_F_numerical_list=[]
  for col in df.columns:
    if df[col].dtype != 'object':
      data_F_numerical_list.append(col)
  data_F_numerical = data_F[data_F_numerical_list]
  return data_F_numerical

In [ ]:
numerical_data(data_F).describe()

,age_conducteur,anciennete_permis_ans,age_vehicule_ans,puissance_fiscale_cv,km_annuels_estimes,coefficient_bonus_malus,nb_sinistres_anterieurs,prime_annuelle_ttc_mad,sinistre_declare,montant_sinistre_mad
count,1000.000000,946.000000,920.000000,970.000000,906.000000,941.000000,1000.000000,1000.000000,1000.000000,1.000000e+03
mean,47.375000,26.394292,9.965217,7.337113,19897.350993,0.957024,0.595000,1160.977240,0.124000,5.035472e+04
std,35.309221,15.872039,7.777292,1.828217,32948.368342,1.483450,0.868176,3135.704927,0.329746,7.056659e+05
min,-5.000000,0.000000,-10.000000,4.000000,-1000.000000,-1.000000,0.000000,-500.000000,0.000000,0.000000e+00
25%,32.750000,13.000000,5.000000,6.000000,10000.000000,0.710000,0.000000,909.120000,0.000000,0.000000e+00
50%,46.000000,27.000000,10.000000,7.000000,15000.000000,0.860000,0.000000,1049.935000,0.000000,0.000000e+00
75%,60.000000,41.000000,15.000000,8.000000,20000.000000,1.000000,1.000000,1204.892500,0.000000,0.000000e+00
max,999.000000,52.000000,100.000000,12.000000,500000.000000,25.000000,4.000000,99999.000000,1.000000,9.999999e+06


In [ ]:
# Age conducteur : valeurs valides entre 18 et 100 ans

mask_age = ~data_F['age_conducteur'].between(18,100)
print(f'🔴 Ages aberrants : {mask_age.sum()} → remplacés par NaN')
data_F.loc[mask_age, 'age_conducteur'] = np.nan

# Age véhicule : entre 0 et 40 ans-

mask_age_vehicule_ans = ~data_F['age_vehicule_ans'].between(0,40) & data_F['age_vehicule_ans'].notna()
print(f'🔴 Age vehicule aberrants : {mask_age_vehicule_ans.sum()} → remplacés par NaN')
data_F.loc[mask_age_vehicule_ans, 'age_vehicule_ans'] = np.nan

# km_annuels : entre 1 000 et 200 000 km
mask_km_annuels_estimes = ~data_F['km_annuels_estimes'].between(1000,200000) & data_F['km_annuels_estimes'].notna()
print(f'🔴 km annuels aberrants : {mask_km_annuels_estimes.sum()} → remplacés par NaN')
data_F.loc[mask_km_annuels_estimes, 'km_annuels_estimes'] = np.nan

# Bonus-malus : entre 0.50 et 3.50

mask_coefficient_bonus_malus = data_F['coefficient_bonus_malus'].notna() & ~data_F['coefficient_bonus_malus'].between(0.50,3.50)
print(f'🔴 Coefficient bonus_malus aberrants : {mask_coefficient_bonus_malus.sum()} → remplacés par NaN')
data_F.loc[mask_coefficient_bonus_malus,'coefficient_bonus_malus'] = np.nan

# Prime : entre 465 et 1648 MAD
Q1 = data_F['prime_annuelle_ttc_mad'].quantile(0.25)
Q3 = data_F['prime_annuelle_ttc_mad'].quantile(0.75)
IQR = Q3 - Q1
inervalle_iqr = [int(Q1-1.5*IQR),int(Q3+1.5*IQR)]
print("l'intervalle estimer pour le prime annuelle par la methode iterquartille est : {}".format(inervalle_iqr))
mask_prime_annuelle_tt = data_F['prime_annuelle_ttc_mad'].notna() & ~data_F['prime_annuelle_ttc_mad'].between(int(Q1-1.5*IQR),int(Q3+1.5*IQR))
print(f'🔴 Prime annuelle aberrants : {mask_prime_annuelle_tt.sum()} → remplacés par NaN')
data_F.loc[mask_prime_annuelle_tt,'prime_annuelle_ttc_mad'] = np.nan




🔴 Ages aberrants : 8 → remplacés par NaN
🔴 Age vehicule aberrants : 5 → remplacés par NaN
🔴 km annuels aberrants : 7 → remplacés par NaN
🔴 Coefficient bonus_malus aberrants : 10 → remplacés par NaN
l'intervalle estimer pour le prime annuelle par la methode iterquartille est : [465, 1648]
🔴 Prime annuelle aberrants : 6 → remplacés par NaN


### 1.6 Correction des incohérences logiques

In [ ]:
# Sinistre=0 mais montant > 0 → incohérence : remettre montant à 0

mask_inc_1 = (data_F['sinistre_declare'] == 0) & (data_F['montant_sinistre_mad'] > 0)
print(f'🔴 Incohérence sinistre=0 & montant>0 : {mask_inc_1.sum()} lignes → montant remis à 0')
data_F.loc[mask_inc_1,'montant_sinistre_mad'] = 0

# Sinistre=1 mais montant = 0 → montant inconnu → NaN

mask_inc_2 = (data_F['sinistre_declare'] == 1) & (data_F['montant_sinistre_mad'] <= 0)
print(f'🔴 Incohérence sinistre=1 & montant=0 : {mask_inc_2.sum()} lignes → montant mis à NaN')
data_F.loc[mask_inc_2,'montant_sinistre_mad'] = np.nan

🔴 Incohérence sinistre=0 & montant>0 : 12 lignes → montant remis à 0
🔴 Incohérence sinistre=1 & montant=0 : 1 lignes → montant mis à NaN


### 1.7 Imputation des valeurs manquantes

In [ ]:
# Variables numériques → médiane (robuste aux outliers)
num_cols =[]
for col in data_F[numerical_data(data_F).columns]:
  if data_F[col].isna().sum() >0:
    num_cols .append(col)
for c in num_cols:
  n_nan = data_F[c].isnull().sum()
  mediane = data_F[c].median()
  data_F[c] = data_F[c].fillna(mediane)
  print(f'✅ {c:35s} → {n_nan} NaN imputés par médiane ({mediane:.2f})')

# Variables catégorielles → mode (valeur la plus fréquente)
cat_cols = ['categorie_socioprofessionnelle']
for col in cat_cols:
    n_nan = data_F[col].isnull().sum()
    if n_nan > 0:
        mode = data_F[col].mode()[0]
        data_F[col] = data_F[col].fillna(mode)
        print(f'✅ {col:35s} → {n_nan} NaN imputés par mode ("{mode}")')

print(f'\n🟢 NaN restants : {data_F.isnull().sum().sum()}')
print(f'🟢 Dataset propre : {data_F.shape}')

✅ age_conducteur                      → 8 NaN imputés par médiane (46.00)
✅ anciennete_permis_ans               → 54 NaN imputés par médiane (27.00)
✅ age_vehicule_ans                    → 85 NaN imputés par médiane (10.00)
✅ puissance_fiscale_cv                → 30 NaN imputés par médiane (7.00)
✅ km_annuels_estimes                  → 101 NaN imputés par médiane (15000.00)
✅ coefficient_bonus_malus             → 69 NaN imputés par médiane (0.86)
✅ prime_annuelle_ttc_mad              → 6 NaN imputés par médiane (1050.34)
✅ montant_sinistre_mad                → 1 NaN imputés par médiane (0.00)
✅ categorie_socioprofessionnelle      → 45 NaN imputés par mode ("Employe")

🟢 NaN restants : 0
🟢 Dataset propre : (1000, 17)


In [ ]:
data_F.isnull().sum()

,0
id_police,0
age_conducteur,0
genre,0
ville,0
categorie_socioprofessionnelle,0
anciennete_permis_ans,0
marque_vehicule,0
age_vehicule_ans,0
puissance_fiscale_cv,0
usage_vehicule,0


In [ ]:
data_F.shape

(1000, 17)